<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step03bCropCowsFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 03b · Cut out the cows: full hour

Takes the tracks from **Step 02d** (SAM 2.1, `Step02dSam2Full/Tracks.csv`) and cuts out the cow over the whole hour: at least 24 crops per kept track and at least one every 15 s (~240 crops, ~10 min). To use the old ByteTrack tracks instead, set `TracksPath` to `Step02bFull/Tracks.csv`. Run Step 03a first if you changed settings.

1. drops duplicate boxes (a weaker box inside a stronger one in the same frame),
2. keeps only tracks that are long and confident enough,
3. picks frames spread over each track, so the crops show different angles,
4. masks each cow with SAM 2.1 (the pen turns grey; a calf touching her, e.g. being licked, can stay in) and saves the crop.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step03bFull/`, **overwriting** each run:

- `KeptTracks.csv`: every track, kept or not and why; `RelativeSize` is box size vs. a typical track (information only)
- `Crops.csv` + `Crops/Track<id>/`: the masked crops
- `Sheets/Overview.jpg` (one crop per kept track) and `Sheets/Track<id>.jpg` (all crops of a track): check these by eye

Use a GPU runtime (**T4**) for SAM. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO/SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.CropCows import CropCows
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
ClipPath = VideoOutputDir / "ReferenceClip.mp4"                 # step 01
TracksPath = VideoOutputDir / "Step02dSam2Full" / "Tracks.csv"  # step 02d (SAM 2); ByteTrack: "Step02bFull"
OutputDir = VideoOutputDir / "Step03bFull"

MinFrames = 10         # drop tracks shorter than this (10 frames = 5 s at 2 fps; use 25 for ByteTrack at 5 fps)
MinConfidence = 0.4    # drop tracks whose average confidence is below this
CropsPerTrack = 24     # at least this many crops per kept track, spread over its time span
CropEverySeconds = 15  # and at least one crop every 15 s (a SAM track can last the whole hour)
MaxSeconds = None      # None = the whole clip
UseMasks = True        # False = plain box crops, no SAM (faster, background kept)

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"
assert TracksPath.exists(), f"Run step 02d first: {TracksPath}"

In [ ]:
# 3. Run: crop + mask, then show which tracks were kept and the overview sheet.
import pandas as pd
from IPython.display import Image, display

Stats = CropCows(ClipPath, TracksPath, OutputDir, MinFrames=MinFrames, MinConfidence=MinConfidence,
                 CropsPerTrack=CropsPerTrack, CropEverySeconds=CropEverySeconds, MaxSeconds=MaxSeconds, UseMasks=UseMasks)

display(pd.read_csv(OutputDir / "KeptTracks.csv"))
display(Image(str(OutputDir / "Sheets" / "Overview.jpg")))
print("Per-track sheets:", OutputDir / "Sheets")

## About this step

**Model:** **SAM 2.1 Small** (Meta's Segment Anything 2.1), given each box as a hint, outlines the animal; everything outside the outline is painted grey.

**Rules:** duplicate boxes removed; tracks need ≥ `MinFrames` frames and average confidence ≥ `MinConfidence`; crops are spread over each track. `RelativeSize` is information only: a cow seen from behind is as small as a calf.

**What we found (whole reference clip, ByteTrack tracks from 02b; to be redone with SAM tracks):** 167 duplicate boxes removed, **14 of 40 tracks kept** (the dropped ones were 1–18-frame fragments), **306 crops, all the same cow**, ~7 min on a T4. When she licks the calf, the calf ends up inside her mask (track 198, ~2560–3220 s). The calf was never kept as its own track.

Track timeline (clip seconds; add 03:35:11 for video time): T2 3–48 standing · T5 51–401 lying · T27 400–481 lying · T37 475–513 lying · T40 513–630 standing · T54 1081–1097 rear · T57 1117–1194 lying · T125 1982–2003 lying · T156 2199–2263 lying · T169 2302–2409 lying · T186 2477–2513 just after the birth · T198 2516–3256 licking the calf · T234 3265–3623 rear · T253 3627–3714 close to the camera.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.